# ML-04 — Search Intelligence Data Contract

My lane (from w01/w02): **Growth / Recovery / Momentum** — predict which pages are about to lose search visibility, then rank them for the editor. This notebook writes down the *data contract* for that lane on the **warehouse slice**: what one row means, which tables and windows I touch, what I predict, what I deliberately exclude — and then proves each claim with a real query on `month=2026-03`.

Simple words, honest numbers. Every claim gets a query next to it.

## 1. Unit of analysis + time window

**What one row means.** In the fact table, one row = **one Search Console observation for one page in one client's account on one calendar day** — `report_date × client_hash_id × content_hash_id`. That is the grain I verify below.

When I build features to *rank pages*, I roll those daily rows up to **one row per page at a decision date** (2026-03-15): the editor gets a ranked queue of pages, not of days.

**Time window.** Everything comes from the mid-panel month `2026-03`, which is inside the snapshot's known-good range (2025-01-27 → 2026-06-30):

- **Slice:** `report_date` in 2026-03-01 … 2026-03-31
- **Feature window:** 2026-03-01 … 2026-03-15 (first half of March, delivered by the daily sync on or before the decision moment)
- **Outcome window:** 2026-03-16 … 2026-03-31 (second half of March, used only to *label*, never as a feature)
- **Decision moment / date:** end of 2026-03-15 — features must be knowable then

In [1]:
# --- Setup: sources, windows, engine ---------------------------------------------
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# work/notebooks -> repo root, wherever the notebook runs from
def find_root():
    p = Path.cwd()
    for _ in range(4):
        if (p / "AGENTS.md").exists():
            return p
        p = p.parent
    return Path(".")

ROOT = find_root()
CACHE = ROOT / "work" / "outputs" / "warehouse_cache"

if not (CACHE / "fact_2026-03.parquet").exists():
    raise SystemExit(
        "Mid-panel month cache not found. Download month=2026-03 plus the two dims "
        "from FlyRank/internship-warehouse into work/outputs/warehouse_cache/ first "
        "(see skills/flyrank/flyrank-data). A READ token is never pasted in a cell."
    )

FACT         = f"read_parquet('{CACHE}/fact_2026-03.parquet')"
DIM_CONTENT  = f"read_parquet('{CACHE}/dim_content.parquet')"
DIM_CLIENTS  = f"read_parquet('{CACHE}/dim_clients.parquet')"

DECISION_DATE = "2026-03-15"
FEAT_LO, FEAT_HI = "2026-03-01", "2026-03-15"
OUT_LO,  OUT_HI  = "2026-03-16", "2026-03-31"

con = duckdb.connect()

print("source cache :", CACHE.name)
print("decision date:", DECISION_DATE)
print("feature window:", FEAT_LO, "->", FEAT_HI)
print("outcome window:", OUT_LO, "->", OUT_HI)

source cache : warehouse_cache
decision date: 2026-03-15
feature window: 2026-03-01 -> 2026-03-15
outcome window: 2026-03-16 -> 2026-03-31


## 2. Fields: feature / label / context / excluded

**Tables I use.** `fact_content_daily_performance` (partition `month=2026-03`) — the daily GSC/GA4 facts. `dim_content` — page metadata (`content_created_date`) joined on `content_hash_id`. `dim_clients` is used only for a coverage check in the limitation section.

**What I predict (label / proxy).** `is_visibility_loss = 1` when a page's **second-half** March impressions are **below** its **first-half** impressions (computed only for pages with ≥ 100 first-half impressions, so tiny pages don't set the label). This is an honest **within-month proxy** for “this page is losing search visibility” — it is *measured from the outcome window, which is never a feature*. The capstone's real label will be a proper future-window decline (e.g. next-30-days vs prior-30-days); same spirit, built carefully later.

**The five features — each is knowable at the decision moment:**

| feature | definition (feature window only) | available when? |
|---|---|---|
| `f_log_impressions` | ln(1 + first-half GSC impressions) | knowable at the decision moment because every input row was delivered by the daily sync on or before 2026-03-15 |
| `f_ctr` | first-half clicks ÷ first-half impressions | knowable at the decision moment because clicks arrive in the same daily sync as impressions, all dated ≤ 2026-03-15 |
| `f_avg_position` | mean GSC position over impression-days in the first half | knowable at the decision moment because it is averaged only on dates ≤ 2026-03-15 |
| `f_active_days` | number of distinct first-half days with ≥ 1 impression | knowable at the decision moment because the 15-day window is already complete at the decision |
| `f_log_age_days` | ln(1 + days between page creation and 2026-03-15) | knowable at the decision moment because page creation date is static metadata known before any march data |

**Deliberate exclusion (one line of why each):**

- **GA4 columns** (`ga4_*`, `sessions_*`, `scroll_events`): my question is about search visibility, and GA4 rows carry a three-valued availability flag (TRUE / FALSE / NULL) with zero-filled early history — mixing a second, messier measurement system into this contract buys noise, not signal.
- **`dim_clients.gsc_data_start` as a feature**: it is context for checking coverage, not a per-page signal.
- **The `_sample` table**: it holds the panel's *last* month and would let me peek at my own test window.
- **Product flags** (`health_score`, `priority_score`, …): they are deliberately absent from this dataset — I must not rebuild them as features.

In [2]:
# --- Build the five-feature frame (feature window only; label comes from outcome window)
feature_sql = f"""
WITH feat AS (
  SELECT client_hash_id, content_hash_id,
    SUM(gsc_impressions) AS f_impressions,
    SUM(gsc_clicks) AS f_clicks,
    AVG(CASE WHEN gsc_impressions > 0 THEN gsc_avg_position END) AS f_avg_position,
    COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0) AS f_active_days
  FROM {FACT}
  WHERE report_date BETWEEN DATE '{FEAT_LO}' AND DATE '{FEAT_HI}'
    AND gsc_data_available IS TRUE
  GROUP BY 1, 2
),
outcome AS (
  SELECT client_hash_id, content_hash_id,
    SUM(gsc_impressions) AS t_impressions
  FROM {FACT}
  WHERE report_date BETWEEN DATE '{OUT_LO}' AND DATE '{OUT_HI}'
    AND gsc_data_available IS TRUE
  GROUP BY 1, 2
)
SELECT
  feat.client_hash_id,
  feat.content_hash_id,
  LN(1 + feat.f_impressions)                                 AS f_log_impressions,
  feat.f_clicks::DOUBLE / NULLIF(feat.f_impressions, 0)      AS f_ctr,
  feat.f_avg_position,
  feat.f_active_days,
  LN(1 + DATEDIFF('day', dim.content_created_date, DATE '{DECISION_DATE}')) AS f_log_age_days,
  CASE WHEN feat.f_impressions >= 100
       THEN (outcome.t_impressions < feat.f_impressions)::INT END AS is_visibility_loss
FROM feat
LEFT JOIN outcome USING (client_hash_id, content_hash_id)
LEFT JOIN {DIM_CONTENT} dim USING (content_hash_id)
"""

frame = con.sql(feature_sql).df()

print("rows in frame (feature window, gsc available):", len(frame))
print("rows with a label (>=100 first-half impressions):", int(frame["is_visibility_loss"].notna().sum()))
print("label rate among labeled rows:", round(frame["is_visibility_loss"].mean(), 4))
print("\nmissingness per feature (fill policy: NaN -> 0 for the demo score):")
for c in ["f_log_impressions", "f_ctr", "f_avg_position", "f_active_days", "f_log_age_days"]:
    print(f"  {c:20s} missing = {int(frame[c].isna().sum()):>6}")
print("\nfirst rows of the frame:")
print(frame.head(3).to_string(index=False))

rows in frame (feature window, gsc available): 151981
rows with a label (>=100 first-half impressions): 77400
label rate among labeled rows: 0.4438

missingness per feature (fill policy: NaN -> 0 for the demo score):
  f_log_impressions    missing =      0
  f_ctr                missing =      0
  f_avg_position       missing =      0
  f_active_days        missing =      0
  f_log_age_days       missing =      0

first rows of the frame:
         client_hash_id          content_hash_id  f_log_impressions  f_ctr  f_avg_position  f_active_days  f_log_age_days  is_visibility_loss
client_73cda7b4e4f265ea content_1e392a54ca96730d           4.343805    0.0       31.565084             15        5.942799                <NA>
client_73cda7b4e4f265ea content_2e7b4f25a1033e4b           3.688879    0.0       28.895238             14        5.942799                <NA>
client_73cda7b4e4f265ea content_2bbf031b7abf488a           4.574711    0.0       52.400661             15        5.978886          

## 3. Verify it with queries (grain, counts, windows, availability)

Three queries, one claim each, all on `month=2026-03`.

In [3]:
# Query 1 — GRAIN: one row really is one (client, content, report_date).
# A non-empty result would prove duplicates in that triple.
q1 = con.sql(f"""
  SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS c
  FROM {FACT}
  GROUP BY 1, 2, 3
  HAVING COUNT(*) > 1
  LIMIT 5
""").df()
print("duplicate content-days found:", len(q1))
q1

duplicate content-days found: 0


,client_hash_id,content_hash_id,report_date,c


In [4]:
# Query 2 — SLICE SIZE + DATE SPAN: how many rows, and over which dates.
q2 = con.sql(f"""
  SELECT COUNT(*)                            AS n_rows,
         MIN(report_date)                     AS first_date,
         MAX(report_date)                     AS last_date,
         COUNT(DISTINCT client_hash_id)       AS n_clients,
         COUNT(DISTINCT content_hash_id)      AS n_content_pages
  FROM {FACT}
""").df()
q2

,n_rows,first_date,last_date,n_clients,n_content_pages
0,9841378,2026-03-01,2026-03-31,55,331437


In [5]:
# Query 3 — AVAILABILITY: filter with IS TRUE, and show why it matters.
# gsc_data_available gates the rows I am allowed to read as real measurements.
q3 = con.sql(f"""
  SELECT
    COUNT(*)                                             AS month_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)   AS gsc_survive_is_true,
    COUNT(*) FILTER (WHERE gsc_data_available IS FALSE)  AS gsc_false,
    COUNT(*) FILTER (WHERE gsc_data_available IS NULL)   AS gsc_null,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)   AS ga4_true,
    COUNT(*) FILTER (WHERE ga4_data_available IS FALSE)  AS ga4_false,
    COUNT(*) FILTER (WHERE ga4_data_available IS NULL)   AS ga4_null
  FROM {FACT}
""").df()
q3
print("surviving my lane's availability filter (gsc IS TRUE):", int(q3["gsc_survive_is_true"].iloc[0]),
      "of", int(q3["month_rows"].iloc[0]))
print("GA4 shows the three-valued case: ", int(q3["ga4_null"].iloc[0]), "NULL rows would be silently dropped by `= TRUE`.")

surviving my lane's availability filter (gsc IS TRUE): 3611061 of 9841378
GA4 shows the three-valued case:  3018741 NULL rows would be silently dropped by `= TRUE`.


## 4. The leakage trap and the honest number

Notebook 02 taught the lesson: `trend_pct` / `trend_direction` were derived from the label's own inputs, so feeding them to the model looked perfect and meant nothing. I replay that lesson **on real warehouse data**: I add **one** label-derived column (`ln(1 + second-half impressions)` — literally the outcome window's defining quantity) and watch a quick score jump toward perfect. Then I delete it and keep the honest number.

In [6]:
FEATURES = ["f_log_impressions", "f_ctr", "f_avg_position", "f_active_days", "f_log_age_days"]

labeled = frame.dropna(subset=["is_visibility_loss"]).copy()
y = labeled["is_visibility_loss"].astype(int).values

# The ONE label-derived column, added on purpose: it re-uses the outcome window.
leak = con.sql(f"""
  SELECT client_hash_id, content_hash_id,
         LN(1 + SUM(gsc_impressions)) AS leak_future_imps
  FROM {FACT}
  WHERE report_date BETWEEN DATE '{OUT_LO}' AND DATE '{OUT_HI}'
    AND gsc_data_available IS TRUE
  GROUP BY 1, 2
""").df()
trap_frame = labeled.merge(leak, on=["client_hash_id", "content_hash_id"], how="left")


def quick_auc(X, y):
    X = pd.DataFrame(X).replace([np.inf, -np.inf], np.nan).fillna(0).values
    return round(cross_val_score(LogisticRegression(max_iter=500, C=1.0), X, y,
                                 cv=3, scoring="roc_auc").mean(), 4)

honest  = quick_auc(labeled[FEATURES], y)
leaky   = quick_auc(trap_frame[FEATURES + ["leak_future_imps"]], y)

print("quick score, 5 honest features only   :", honest)
print("quick score, 5 features + 1 leak column:", leaky, " <-- near-perfect, the trap")

# ...and delete it. The honest number survives.
trap_frame.drop(columns=["leak_future_imps"], inplace=True)
print("\nleak column still present?", "leak_future_imps" in trap_frame.columns)
print("honest number after removing the leak  :", quick_auc(trap_frame[FEATURES], y))

quick score, 5 honest features only   : 0.6262
quick score, 5 features + 1 leak column: 1.0  <-- near-perfect, the trap

leak column still present? False
honest number after removing the leak  : 0.6262


### One named limitation of this slice

**The label is a within-March proxy, not a true future-window label.** `is_visibility_loss` compares the second half of March to the first half of March — it is *measured*, not *predicted*, and it says nothing yet about whether today's features predict *next month's* decline. The capstone label must be built as `prior window → future window` and validated with time-aware splits.

A second, measurement-level caution (checked below): per-client GSC history does **not** start together — 37 of 104 clients have no `gsc_data_start`, and only 52 cover all of March 2026. So the March slice is unbalanced across clients: a client with no GSC data contributes nothing to this frame, and equal calendar windows do **not** mean equal history depth.

In [7]:
# Limitation check: per-client GSC coverage inside the slice window.
lim = con.sql(f"""
  SELECT
    COUNT(*)                                                       AS clients_total,
    COUNT(*) FILTER (WHERE gsc_data_start <= DATE '{FEAT_LO}')     AS clients_cover_all_march,
    COUNT(*) FILTER (WHERE gsc_data_start IS NULL)                 AS clients_no_gsc_start,
    MIN(gsc_data_start)                                            AS earliest_gsc_start,
    MAX(gsc_data_start)                                            AS latest_gsc_start
  FROM {DIM_CLIENTS}
""").df()
lim

,clients_total,clients_cover_all_march,clients_no_gsc_start,earliest_gsc_start,latest_gsc_start
0,104,52,37,2025-01-27,2026-06-02


## Self-check

Before submitting, confirm each line honestly:

- [x] Every section is filled — plain-words contract (5 answers) folded into §1–§2, plus markdown thinking AND the code that backs it
- [x] Exactly three verification queries (§3) with outputs visible — grain, slice count + date span, availability via `IS TRUE`
- [x] The five-feature frame has an “available when?” line per feature (§2 table)
- [x] The deliberate-leak experiment is shown, then removed, and the honest number kept (§4)
- [x] One named limitation of the slice (§4)
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — only pseudonymized hash ids
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit my repo URL on the card. Done.